# 04 · Analyze runs (post-hoc)

Aggregates every `runs/*/result.json` into one table and contrasts **random-split
vs LOSO** (spec §6.3) — the robustness story. Also surfaces the design-health flag
`uncertainty_ok` (S(wrong) ≤ S(correct), spec §6.2). Re-run after training fills `runs/`.

In [ ]:
import os, sys, json, glob
from pathlib import Path
_root = Path.cwd()
while not (_root / "data" / "metadata").exists():
    _root = _root.parent
os.chdir(_root); sys.path.insert(0, str(_root))
import pandas as pd

rows = []
for f in sorted(glob.glob("runs/*/result.json")):
    d = json.load(open(f))
    m = d.get("test_metrics", {}); mon = d.get("test_monitor", {})
    rows.append({
        "run": Path(f).parent.name, "split": d["split_file"], "fold": d.get("fold_id"),
        "seed": d["seed"], "n_test": d.get("n_test"),
        "auc": m.get("auc"), "auprc": m.get("auprc"), "nll": m.get("nll"),
        "ece": m.get("ece"), "median_S": m.get("median_S"),
        "uncertainty_ok": mon.get("uncertainty_ok"),
    })
df = pd.DataFrame(rows)
print(f"{len(df)} run(s) found under runs/")
df

In [ ]:
if len(df):
    df["kind"] = df["split"].map(lambda s: "random" if "random" in s else "LOSO/aux")
    print("random-split vs LOSO (mean AUC / NLL):")
    display(df.groupby("kind")[["auc", "nll", "ece", "median_S"]].mean().round(3))
    bad = df[df["uncertainty_ok"] == False]
    print("runs where S(wrong) > S(correct) (design concern):",
          bad["run"].tolist() if len(bad) else "none")
else:
    print("no runs yet — execute 03_train (via scripts/hpc/run_all_folds.sh) first")